In [55]:
import pandas as pd
import numpy as np
import re
import os

## Скачиваем данные

In [56]:
data_train = pd.read_csv('./data/train.csv')
data_test = pd.read_csv('./data/test.csv')

In [57]:
data_train.head()

,ID,url,title,label
0,0,m.kp.md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0
1,1,www.kp.by,Эта песня стала известна многим телезрителям б...,0
2,2,fanserials.tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0
3,3,colorbox.spb.ru,Не Беси Меня Картинки,0
4,4,tula-sport.ru,В Новомосковске сыграют следж-хоккеисты алекси...,0


In [58]:
data_test.head()

,ID,url,title
0,135309,www.kommersant.ru,Шестой кассационный суд в Самаре начнет работу...
1,135310,urexpert.online,"Что такое индексация алиментов, кем и в каких ..."
2,135311,imperimeha.ru,Женщинам | Империя Меха - Part 12
3,135312,national-porn.com,"Небритые, волосатые киски: Порно всех стран и ..."
4,135313,2gis.ru,67


In [59]:
data_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135309 entries, 0 to 135308
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   ID      135309 non-null  int64 
 1   url     135309 non-null  object
 2   title   135308 non-null  object
 3   label   135309 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 4.1+ MB


один пропуск в title

In [60]:
data_train.describe()

,ID,label
count,135309.000000,135309.000000
mean,67654.000000,0.123532
std,39060.488124,0.329048
min,0.000000,0.000000
25%,33827.000000,0.000000
50%,67654.000000,0.000000
75%,101481.000000,0.000000
max,135308.000000,1.000000


## Обрабатываем данные

In [61]:
def clean_url(url):
    url = url.lower()
    url = re.sub(r'https?://', ' ', url)
    url = re.sub(r'www\.', ' ', url)
    url = re.sub(r'[/._\-?=&]', ' ', url)
    return url

In [62]:
def clean_text(text):
    text = text.lower()
    tokens = text.split()

    cleaned_tokens = []
    for token in tokens:
        if len(token) <= 1:
            continue
        cleaned_tokens.append(token)

    return ' '.join(cleaned_tokens)

In [63]:
def build_text(df):
    df = df.copy()
    df['url'] = df['url'].apply(clean_url)
    df['text'] = (df['url'] + ' ' + df['title'].astype(str)).apply(clean_text)
    return df

In [64]:
data_train = data_train.dropna(subset=['title']).copy()
data_train = data_train.drop_duplicates(subset=['title']).reset_index(drop=True)

In [65]:
data_train = build_text(data_train)
data_test = build_text(data_test)

In [66]:
data_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129302 entries, 0 to 129301
Data columns (total 5 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   ID      129302 non-null  int64 
 1   url     129302 non-null  object
 2   title   129302 non-null  object
 3   label   129302 non-null  int64 
 4   text    129302 non-null  object
dtypes: int64(2), object(3)
memory usage: 4.9+ MB


In [67]:
data_train.head()

,ID,url,title,label,text
0,0,m kp md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0,kp md экс-министр экономики молдовы главе мидэ...
1,1,kp by,Эта песня стала известна многим телезрителям б...,0,kp by эта песня стала известна многим телезрит...
2,2,fanserials tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0,fanserials tv банши сезон серия бремя красоты ...
3,3,colorbox spb ru,Не Беси Меня Картинки,0,colorbox spb ru не беси меня картинки
4,4,tula sport ru,В Новомосковске сыграют следж-хоккеисты алекси...,0,tula sport ru новомосковске сыграют следж-хокк...


In [68]:
data_train['text'][:10]

0    kp md экс-министр экономики молдовы главе мидэ...
1    kp by эта песня стала известна многим телезрит...
2    fanserials tv банши сезон серия бремя красоты ...
3                colorbox spb ru не беси меня картинки
4    tula sport ru новомосковске сыграют следж-хокк...
5    beregifiguru ru салат: корейская морковь копче...
6    ekb vseinstrumenti ru угловой пневмогайковерт ...
7    mirtabaka org табачный магазин `мир табака` ку...
8    xlecx com league of legends page porn comics f...
9    bus biletyplus ua 08:00 (4766р), одесса ростов...
Name: text, dtype: object

## Подготовка перед обучением

In [69]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from scipy.sparse import hstack

In [70]:
sss = StratifiedShuffleSplit(n_splits=50, test_size=0.008, random_state=42)

In [71]:
os.makedirs('answers', exist_ok=True)

## Обучение

In [72]:
for fold, (train_index, valid_index) in enumerate(sss.split(data_train['text'], data_train['label']), 1):
    X_train = data_train['text'].iloc[train_index]
    y_train = data_train['label'].iloc[train_index]

    X_valid = data_train['text'].iloc[valid_index]
    y_valid = data_train['label'].iloc[valid_index]

    word_vectorizer = TfidfVectorizer(
        analyzer='word',
        ngram_range=(2, 5),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True
    )

    char_vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=(3, 5),
        min_df=2
    )

    X_train_word = word_vectorizer.fit_transform(X_train)
    X_valid_word = word_vectorizer.transform(X_valid)
    X_test_word = word_vectorizer.transform(data_test['text'])

    X_train_char = char_vectorizer.fit_transform(X_train)
    X_valid_char = char_vectorizer.transform(X_valid)
    X_test_char = char_vectorizer.transform(data_test['text'])

    X_train_vect = hstack([X_train_word, X_train_char])
    X_valid_vect = hstack([X_valid_word, X_valid_char])
    X_test_vect = hstack([X_test_word, X_test_char])

    model = LogisticRegression(
        max_iter=20000,
        class_weight='balanced',
        solver='saga',
        penalty='l2',
        C=100000,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train_vect, y_train)
    y_proba = model.predict_proba(X_valid_vect)[:, 1]

    best_f1 = 0
    best_threshold = 0.5
    thresholds = np.linspace(0.01, 0.99, 200)
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_valid, y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = t

    print(f'{fold}) F1 = {best_f1:.6f}')
    print()

    test_proba = model.predict_proba(X_test_vect)[:, 1]
    test_pred = (test_proba >= best_threshold).astype(int)

    answer = pd.DataFrame({
        'ID': data_test['ID'],
        'label': test_pred
    })

    filename = f'answers/answer{fold}.csv'
    answer.to_csv(filename, index=False)


1) F1 = 0.992308

2) F1 = 0.992366

3) F1 = 0.992308

4) F1 = 0.996198

5) F1 = 0.988506

6) F1 = 0.992424

7) F1 = 0.988679

8) F1 = 0.996169

9) F1 = 0.992308

10) F1 = 0.996198

11) F1 = 0.992366

12) F1 = 1.000000

13) F1 = 0.992308

14) F1 = 0.988593

15) F1 = 1.000000

16) F1 = 0.992424

17) F1 = 0.992308

18) F1 = 0.996169

19) F1 = 0.992366

20) F1 = 0.984848

21) F1 = 0.984615

22) F1 = 0.992366

23) F1 = 0.988593

24) F1 = 0.984615

25) F1 = 0.988593

26) F1 = 0.988506

27) F1 = 0.996198

28) F1 = 0.992366

29) F1 = 0.996198

30) F1 = 1.000000

31) F1 = 0.996198

32) F1 = 0.992366

33) F1 = 0.992424

34) F1 = 0.988593

35) F1 = 0.992308

36) F1 = 0.996198

37) F1 = 0.996198

38) F1 = 0.992366

39) F1 = 0.988417

40) F1 = 0.996198

41) F1 = 0.996169

42) F1 = 0.992366

43) F1 = 0.996169

44) F1 = 0.988506

45) F1 = 0.988593

46) F1 = 0.992424

47) F1 = 0.992308

48) F1 = 0.996169

49) F1 = 0.984848

50) F1 = 1.000000

